In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim
from torchvision.datasets import ImageFolder
from PIL import Image
import pickle
import numpy as np
from skorch import NeuralNetClassifier # Use PyTorch Models in scikit-learn
from sklearn.model_selection import GridSearchCV # Grid Search

In [2]:
# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print(device)

cuda


# Data

In [3]:
# Load the dataset from the pickle file
file_path = "C:/Users/101194208/Desktop/Die-Cast Ensemble/Data/data_oversample_usingViT.pkl"

with open(file_path, 'rb') as file:
    loaded_data = pickle.load(file)

# Extract image arrays and labels
X_train, X_val, X_test = loaded_data['var1'], loaded_data['var2'], loaded_data['var3']
y_train, y_val, y_test = loaded_data['var4'], loaded_data['var5'], loaded_data['var6']

# Ensure data is in NumPy array format
if isinstance(X_train, list):
    X_train, X_val, X_test = np.array(X_train), np.array(X_val), np.array(X_test)
    y_train, y_val, y_test = np.array(y_train), np.array(y_val), np.array(y_test)

print(f"Train Data Shape: {X_train.shape}, Labels Shape: {y_train.shape}")
print(f"Validation Data Shape: {X_val.shape}, Labels Shape: {y_val.shape}")
print(f"Test Data Shape: {X_test.shape}, Labels Shape: {y_test.shape}")

Train Data Shape: (4854, 3, 500, 500), Labels Shape: (4854,)
Validation Data Shape: (606, 3, 500, 500), Labels Shape: (606,)
Test Data Shape: (334, 3, 500, 500), Labels Shape: (334,)


In [4]:
import cv2
import numpy as np

new_height = 224
new_width = 224

data = X_train
resized_data = np.empty((data.shape[0], data.shape[1], new_height, new_width), dtype=data.dtype)

for i in range(data.shape[0]):
    # Extract a single 3D image (channels, height, width)
    img_3d = data[i]

    # Transpose to (height, width, channels) for OpenCV
    img_cv2 = np.transpose(img_3d, (1, 2, 0))

    # Resize the image
    resized_img_cv2 = cv2.resize(img_cv2, (new_width, new_height), interpolation=cv2.INTER_LINEAR)

    # Transpose back to (channels, height, width)
    resized_3d = np.transpose(resized_img_cv2, (2, 0, 1))

    # Store in the new array
    resized_data[i] = resized_3d

print(resized_data.shape)

(4854, 3, 224, 224)


In [5]:
X_train = torch.Tensor(resized_data)
y_train = torch.LongTensor(y_train)

# Model Construction

In [6]:
import torch.nn as nn
import torchvision.models as models
from torchvision.models import ViT_B_16_Weights

# Load pre-trained Vision Transformer (ViT-B/16)
vit_model = models.vision_transformer.vit_b_16(weights=ViT_B_16_Weights.DEFAULT)

# Modify the classifier head for 3 classes
num_features = vit_model.heads.head.in_features
vit_model.heads.head = nn.Linear(num_features, 3)  # 3 classes: good, crack, cf

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vit_model = vit_model.to(device)

# # Define loss function and optimizer
# criterion = nn.CrossEntropyLoss()
# optimizer = torch.optim.Adam(vit_model.parameters(), lr=0.0001)

# Create Model with skorch

In [7]:
model = NeuralNetClassifier(
    vit_model,
    criterion = nn.CrossEntropyLoss(),
    optimizer = optim.Adam,
    verbose = False,
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu'))

# Grid Search

In [8]:
param_grid = {
    'batch_size': [32, 64],
    'max_epochs': [20, 30, 40],
    'optimizer__lr': [0.0001, 0.001, 0.01, 0.1]}

In [9]:
%%time

grid = GridSearchCV(estimator=model, param_grid=param_grid, n_jobs=1, cv=3)
grid_result = grid.fit(X_train, y_train)

CPU times: total: 1d 16h 28min 44s
Wall time: 5h 37min 32s


In [10]:
# summarize results
print("Best: %f using %s" % (grid_result.best_score_, grid_result.best_params_))
means = grid_result.cv_results_['mean_test_score']
stds = grid_result.cv_results_['std_test_score']
params = grid_result.cv_results_['params']
for mean, stdev, param in zip(means, stds, params):
    print("%f (%f) with: %r" % (mean, stdev, param))

Best: 0.573548 using {'batch_size': 64, 'max_epochs': 40, 'optimizer__lr': 0.0001}
0.333333 (0.000291) with: {'batch_size': 32, 'max_epochs': 20, 'optimizer__lr': 0.0001}
0.333333 (0.000291) with: {'batch_size': 32, 'max_epochs': 20, 'optimizer__lr': 0.001}
0.333333 (0.000291) with: {'batch_size': 32, 'max_epochs': 20, 'optimizer__lr': 0.01}
0.333333 (0.000291) with: {'batch_size': 32, 'max_epochs': 20, 'optimizer__lr': 0.1}
0.463123 (0.098566) with: {'batch_size': 32, 'max_epochs': 30, 'optimizer__lr': 0.0001}
0.333333 (0.000291) with: {'batch_size': 32, 'max_epochs': 30, 'optimizer__lr': 0.001}
0.333333 (0.000291) with: {'batch_size': 32, 'max_epochs': 30, 'optimizer__lr': 0.01}
0.333333 (0.000291) with: {'batch_size': 32, 'max_epochs': 30, 'optimizer__lr': 0.1}
0.377009 (0.062058) with: {'batch_size': 32, 'max_epochs': 40, 'optimizer__lr': 0.0001}
0.333333 (0.000291) with: {'batch_size': 32, 'max_epochs': 40, 'optimizer__lr': 0.001}
0.333333 (0.000291) with: {'batch_size': 32, 'max_